In [2]:
!pip install requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 KB 959.6 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.3/133.3 KB 2.0 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 KB 8.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 KB 386.4 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 KB 1.9 MB/s eta 0:00:0000:0100:01


In [3]:
import requests

In [4]:
response = requests.get(" https://www.bayut.bh/en/to-rent/commercial/bahrain/")


In [5]:
response

<Response [200]>

In [6]:
print(response.headers)

{'Content-Type': 'text/html;charset=utf-8', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Date': 'Thu, 18 Jun 2026 09:52:28 GMT', 'Server': 'nginx', 'Set-Cookie': 'anonymous_session_id=mqjbmsxt0mu6jwrh; Domain=bayut.bh; Path=/; Secure, device_id=mqjbmsxtt11ddjjg; Max-Age=31536000; Domain=bayut.bh; Path=/; Expires=Fri, 18 Jun 2027 09:52:27 GMT; Secure', 'Cache-Control': 'max-age=1200', 'Vary': 'Accept-Encoding, Cookie', 'Content-Security-Policy': "frame-ancestors 'self'", 'Content-Encoding': 'gzip', 'Strict-Transport-Security': 'max-age=63072000; includeSubDomains', 'X-Cache': 'Miss from cloudfront', 'Via': '1.1 7438fda4fba8f76d9be49a9dfb777144.cloudfront.net (CloudFront)', 'X-Amz-Cf-Pop': 'MAA51-P2', 'Alt-Svc': 'h3=":443"; ma=86400', 'X-Amz-Cf-Id': 'vM_fb04CGtbLfy9_JFZgycQl2kpKTZsNBk5SducO9xbZcqe6HL8P-A=='}


In [19]:
!pip install parsel

  Using cached lxml-6.1.1-cp310-cp310-manylinux_2_26_x86_64.manylinux_2_28_x86_64.whl (5.3 MB)


In [20]:
import requests
from parsel import Selector

class BayutScraper:

    def __init__(self):
        self.found_count = 0
        self.current_page = 1

    def get_page(self, url):

        response = requests.get(url)

        return Selector(text=response.text)

    def parse(self):

        url = "https://www.bayut.bh/en/to-rent/commercial/bahrain/"

        selector = self.get_page(url)

        property_urls = selector.xpath(
            '//a[contains(@href, "/en/property/details-")]/@href'
        ).getall()

        for p_url in property_urls:

            self.scrape_details(
                "https://www.bayut.bh" + p_url
            )

    def scrape_details(self, url):

        selector = self.get_page(url)

        title = selector.xpath('//h1/text()').get()

        price = selector.xpath(
            '//span[@aria-label="Price"]//text()'
        ).get()

        print(title, price)


scraper = BayutScraper()

scraper.parse()

1 Bedroom Other Commercial For Rent Gufool, Capital Governorate 350
1 Bedroom Other Commercial For Rent Gufool, Capital Governorate 350
1 Commercial Space For Rent in Jid Ali, Capital Governorate 300
1 Commercial Space For Rent in Jid Ali, Capital Governorate 300
1 Other Commercial For Rent Manama 800
1 Other Commercial For Rent Manama 800
1 Other Commercial For Rent Manama 800
1 Bedroom Other Commercial For Rent in Mahooz, Capital Governorate 1,000
1 Bedroom Other Commercial For Rent in Mahooz, Capital Governorate 1,000
Studio Other Commercial For Rent Burhama, Capital Governorate 300
Studio Other Commercial For Rent Burhama, Capital Governorate 300
Studio Other Commercial For Rent Burhama, Capital Governorate 300


In [30]:
import csv
import time
import random
import requests
from lxml import html

class BayutSpiderEquivalent:
    def __init__(self, max_items=1050):
        self.name = "bayut_spider"
        self.base_url = "https://www.bayut.bh"
        self.first_page_url = "https://www.bayut.bh/en/to-rent/commercial/bahrain/"
        self.pagination_template = "https://www.bayut.bh/en/to-rent/commercial/bahrain/page-{}/"
        
        self.found_count = 0
        self.current_page = 1 
        self.max_items = max_items
        self.scraped_items = []

        # Create a persistent session instance to mimic a real browser tab
        self.session = requests.Session()

        # Headers attached permanently to the session
        self.session.headers.update({
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.9",
            "Referer": "https://www.google.com/",
            "Connection": "keep-alive"
        })

    def fetch_tree(self, url):
        """Uses the persistent session to retain browser identity and cookies."""
        try:
            response = self.session.get(url, timeout=15)
            if response.status_code == 200:
                return html.fromstring(response.content)
            else:
                print(f"[{self.name}] Failed to fetch {url}. Code: {response.status_code}")
        except Exception as e:
            print(f"[{self.name}] Connection issue requesting {url}: {e}")
        return None

    def parse(self, tree):
        """Processes directory list view, pulling property links just like Scrapy's parse."""
        if tree is None:
            return []
        property_urls = tree.xpath('//a[contains(@href, "/en/property/details-")]/@href')
        return list(set(property_urls))

    def scrape_details(self, response_url, tree):
        """Extracts individual data items using your exact field configurations."""
        if tree is None:
            return None

        self.found_count += 1
        listing_data = {}

        if self.found_count % 10 == 0:
            print(f"[{self.name}] Progress: {self.found_count} items scraped.")

        listing_data['url'] = response_url
        
        ref_text = tree.xpath('string(//span[@aria-label="Reference"])') or None
        listing_data['reference_number'] = ref_text
        
        if ref_text and 'ID' in ref_text:
            listing_data['id'] = ref_text.split('ID')[-1].strip()
        else:
            listing_data['id'] = None

        listing_data['purpose'] = tree.xpath('string(//span[@aria-label="Purpose"])') or None
        listing_data['title'] = tree.xpath('string(//h1)') or None
        
        desc_parts = tree.xpath('//span[@aria-label="Description"]//text() | //div[@aria-label="Property description"]//text()')
        listing_data['description'] = " ".join([p.strip() for p in desc_parts if p.strip()])
        
        listing_data['location'] = tree.xpath('string(//div[@aria-label="Property header"])') or None
        listing_data['price'] = tree.xpath('string(//span[@aria-label="Price"])') or None
        listing_data['currency'] = tree.xpath('string(//span[@aria-label="Currency"])') or None
        listing_data['price_per'] = tree.xpath('string(//span[@aria-label="Frequency"])') or None
        listing_data['furnished'] = tree.xpath('string(//span[@aria-label="Furnishing"])') or None
        
        amenity_nodes = tree.xpath('//span[@aria-label="Amenity"]//text()')
        listing_data['amenities'] = ", ".join(set([a.strip() for a in amenity_nodes if a.strip()]))
        
        listing_data['details'] = tree.xpath('string(//span[@aria-label="Area"])') or None
        listing_data['agent_name'] = tree.xpath('string(//span[@aria-label="Agent name"])') or None
        
        all_pics = tree.xpath('//picture//img/@src')
        listing_data['property_image_urls'] = list(set(all_pics)) 
        listing_data['number_of_photos'] = len(listing_data['property_image_urls'])
        
        nav_crumbs = tree.xpath('//div[@aria-label="Breadcrumb"]//span/text()')
        listing_data['breadcrumb'] = " > ".join([n.strip() for n in nav_crumbs if n.strip()])
        
        listing_data['property_type'] = tree.xpath('string(//span[@aria-label="Type"])') or None

        return listing_data

    def run(self):
        print(f"[{self.name}] Initializing run execution...")

        while self.found_count < self.max_items:
            if self.current_page == 1:
                next_page_url = self.first_page_url
            else:
                next_page_url = self.pagination_template.format(self.current_page)
                
            print(f"[{self.name}] Navigating to Page {self.current_page}: {next_page_url}...")
            
            main_tree = self.fetch_tree(next_page_url)
            p_urls = self.parse(main_tree)
            
            if not p_urls:
                print(f"[{self.name}] No valid links identified on this index page.")
                break

            for p_url in p_urls:
                if self.found_count >= self.max_items:
                    break
                
                if p_url.startswith("/"):
                    p_url = self.base_url + p_url
                
                # Update Referer dynamically to simulate regular site activity
                self.session.headers.update({"Referer": next_page_url})
                
                detail_tree = self.fetch_tree(p_url)
                item = self.scrape_details(p_url, detail_tree)
                
                if item:
                    self.scraped_items.append(item)
                    print(f" Scraped property code: {item['id']}")
                    
                time.sleep(random.uniform(2.0, 4.0))

            if self.found_count < self.max_items:
                self.current_page += 1
                self.session.headers.update({"Referer": next_page_url})
                time.sleep(5) 
            else:
                break

        print(f"[{self.name}] Finalized crawl execution. Gathered {len(self.scraped_items)} datasets successfully.")
        return self.scraped_items


# --- Execution and CSV Exporter ---
if __name__ == "__main__":
    # Create instance to scrape 15 items across pagination steps
    spider = BayutSpiderEquivalent(max_items=15)
    all_extracted_records = spider.run()
    
    # Check if we have results to write down
    if all_extracted_records:
        filename = "bayut_commercial_rent.csv"
        print(f"\n Saving data to file: {filename}...")
        
        # Get all field names automatically from dictionary keys
        column_headers = all_extracted_records[0].keys()
        
        # Write structural rows out into standard formatting layout
        with open(filename, mode='w', newline='', encoding='utf-8') as output_file:
            dict_writer = csv.DictWriter(output_file, fieldnames=column_headers)
            
            # Create header structure row
            dict_writer.writeheader()
            # Push compiled record dict lists to row indexes
            dict_writer.writerows(all_extracted_records)
            
        print(f"Success! The spreadsheet file '{filename}' has been saved in this folder.")
    else:
        print(" No datasets gathered. File generation skipped.")

[bayut_spider] Initializing run execution...
[bayut_spider] Navigating to Page 1: https://www.bayut.bh/en/to-rent/commercial/bahrain/...
 Scraped property code: 105667506
 Scraped property code: 105704703
 Scraped property code: 105648594
 Scraped property code: 105690991
 Scraped property code: 105697030
[bayut_spider] Navigating to Page 2: https://www.bayut.bh/en/to-rent/commercial/bahrain/page-2/...
[bayut_spider] Failed to fetch https://www.bayut.bh/en/to-rent/commercial/bahrain/page-2/. Code: 404
[bayut_spider] No valid links identified on this index page.
[bayut_spider] Finalized crawl execution. Gathered 5 datasets successfully.

 Saving data to file: bayut_commercial_rent.csv...
Success! The spreadsheet file 'bayut_commercial_rent.csv' has been saved in this folder.
